In [1]:
!pip install torch transformers numpy

In [2]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)
model.eval()

print(f"Model loaded: {model_name}")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded: gpt2


In [3]:
# Get the hidden size of the model
hidden_size = model.config.n_embd  # 768 for GPT-2

# A random steering vector (in real use, compute from data!)
torch.manual_seed(42)
steering_vector = torch.randn(hidden_size)
steering_vector = steering_vector / steering_vector.norm()  # normalize

print(f"Steering vector shape: {steering_vector.shape}")

Steering vector shape: torch.Size([768])


In [4]:
def get_hidden_state(prompt, layer=6):
    """Get the hidden state from a specific layer for a prompt."""
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    # hidden_states[layer] shape: (batch, seq_len, hidden_size)
    return outputs.hidden_states[layer][0, -1, :]  # last token

# Example: steer toward "positive sentiment"
positive = get_hidden_state("I love this, it's wonderful and")
negative = get_hidden_state("I hate this, it's terrible and")

steering_vector = positive - negative
steering_vector = steering_vector / steering_vector.norm()

In [8]:
class SteeringHook:
    def __init__(self, steering_vector, alpha, layer_idx):
        self.steering_vector = steering_vector
        self.alpha = alpha
        self.layer_idx = layer_idx

    def __call__(self, module, input, output):
        # Case 1: output is a Tensor (GPT-2 blocks do this)
        if isinstance(output, torch.Tensor):
            return output + self.alpha * self.steering_vector

        # Case 2: output is a tuple (some models)
        hidden_states = output[0]
        steered = hidden_states + self.alpha * self.steering_vector
        return (steered,) + tuple(output[1:])

In [9]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# ---------- 1. Load model ----------
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()

# ---------- 2. Build steering vector (contrastive) ----------
def get_hidden(prompt, layer=6):
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        out = model(**inputs, output_hidden_states=True)
    return out.hidden_states[layer][0, -1, :]  # last token, that layer

pos = get_hidden("This is amazing and wonderful and")
neg = get_hidden("This is awful and terrible and")
steering_vector = pos - neg
steering_vector = steering_vector / steering_vector.norm()
print("Steering vector ready. Norm =", steering_vector.norm().item())

# ---------- 3. Hook ----------
class SteeringHook:
    def __init__(self, vec, alpha):
        self.vec, self.alpha = vec, alpha
    def __call__(self, module, inp, out):
        # GPT-2 blocks return a Tensor (not a tuple)
        if isinstance(out, torch.Tensor):
            return out + self.alpha * self.vec
        # Fallback for tuple outputs
        return (out[0] + self.alpha * self.vec,) + tuple(out[1:])

# ---------- 4. Generate with steering ----------
def generate_with_steering(prompt, alpha=0.0, layer=6, max_new_tokens=30):
    handle = model.transformer.h[layer].register_forward_hook(
        SteeringHook(steering_vector, alpha)
    )
    try:
        inputs = tokenizer(prompt, return_tensors="pt")
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        return tokenizer.decode(out[0], skip_special_tokens=True)
    finally:
        handle.remove()

# ---------- 5. Run ----------
prompt = "The weather today is"
for alpha in [-10.0, 0.0, 10.0, 20.0]:
    print(f"\nα={alpha:+.0f}: {generate_with_steering(prompt, alpha=alpha)}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Steering vector ready. Norm = 1.0

α=-10: The weather today is very bad. We're going to have to get out of here. We're going to have to get out of here. We're going to have

α=+0: The weather today is very good, and we're going to be able to get some good weather tomorrow.

"We're going to be able to get some good

α=+10: The weather today is good, but I'm not sure if it's going to be a good day for the snow."

The snow is expected to fall in the

α=+20: The weather today is good, and I'm sure you'll be able to find a few more of them.

The next day, I was able to find a


In [10]:
for alpha in [0.0, 50.0, -50.0]:
    print(f"\nα={alpha:+.0f}:\n{generate_with_steering(prompt, alpha=alpha)}")


α=+0:
The weather today is very good, and we're going to be able to get some good weather tomorrow.

"We're going to be able to get some good

α=+50:
The weather today is a bit of a surprise.

The weather is a bit of a surprise.

The weather is a bit of a surprise.



α=-50:
The weather today is worse than ever, and we are in the most dangerous moment in our world, and we are in the most urgent moment in our world, and we


In [ ]:
import matplotlib.pyplot as plt

# Try different strengths
alphas = [-10, -5, 0, 5, 10, 15]
results = [generate_with_steering(prompt, alpha=a) for a in alphas]

for a, r in zip(alphas, results):
    print(f"\nα = {a:+d}:")
    print(r)